# Download Real TLC Green Taxi Data

Download real adjacent-month NYC TLC Green Taxi parquet files. The Ray capstone requires adjacent months, so this notebook uses January 2023 as the reference month and February 2023 as the replay month.

Imports and constants used by this notebook.

In [1]:
from __future__ import annotations
import os
from pathlib import Path
from urllib.request import urlretrieve
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
DEFAULT_MONTHS = ("2023-01", "2023-02")


Define the `file_name` function.

In [2]:
def file_name(month: str) -> str:
    return f"green_tripdata_{month}.parquet"


Define the `download_month` function.

In [3]:
def download_month(month: str, output_dir: Path) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    name = file_name(month)
    url = f"{BASE_URL}/{name}"
    path = output_dir / name
    if path.exists() and path.stat().st_size > 0:
        print(f"Already exists: {path} ({path.stat().st_size:,} bytes)")
        return path
    print(f"Downloading {url}")
    urlretrieve(url, path)
    print(f"Wrote {path} ({path.stat().st_size:,} bytes)")
    return path


Define the `download_data` function.

In [4]:
def download_data(output_dir: Path, months: list[str]) -> list[Path]:
    paths = [download_month(month, output_dir) for month in months]
    print("\nDone. Files:")
    for path in paths:
        print(f"- {path} ({path.stat().st_size:,} bytes)")
    return paths


Download the real TLC parquet files.

This writes the downloaded parquet files to `data` by default. In the Docker-cluster run, `CAPSTONE_DATA_DIR` points to a mounted path so the data remains visible after the Ray job finishes.

In [5]:
data_dir = Path(os.getenv("CAPSTONE_DATA_DIR", "data"))
download_data(data_dir, ["2023-01", "2023-02"])


Already exists: data\green_tripdata_2023-01.parquet (1,427,002 bytes)
Already exists: data\green_tripdata_2023-02.parquet (1,533,740 bytes)

Done. Files:
- data\green_tripdata_2023-01.parquet (1,427,002 bytes)
- data\green_tripdata_2023-02.parquet (1,533,740 bytes)


[WindowsPath('data/green_tripdata_2023-01.parquet'),
 WindowsPath('data/green_tripdata_2023-02.parquet')]

Inspect the downloaded files.

In [6]:
import json
import pandas as pd

reference_path = data_dir / "green_tripdata_2023-01.parquet"
replay_path = data_dir / "green_tripdata_2023-02.parquet"
reference = pd.read_parquet(reference_path)
replay = pd.read_parquet(replay_path)
download_manifest = {
    "data_dir": str(data_dir),
    "reference_file": str(reference_path),
    "reference_rows": int(len(reference)),
    "reference_size_mb": round(reference_path.stat().st_size / (1024 * 1024), 2),
    "replay_file": str(replay_path),
    "replay_rows": int(len(replay)),
    "replay_size_mb": round(replay_path.stat().st_size / (1024 * 1024), 2),
}
manifest_path = data_dir / "download_manifest.json"
manifest_path.write_text(json.dumps(download_manifest, indent=2), encoding="utf-8")

print("Download evidence")
print("-----------------")
print(f"reference rows: {len(reference)}")
print(f"replay rows: {len(replay)}")
print(f"download manifest saved to: {manifest_path}")
display(reference[["lpep_pickup_datetime", "PULocationID"]].head())
display(replay[["lpep_pickup_datetime", "PULocationID"]].head())


reference rows: 68211
replay rows: 64809


,lpep_pickup_datetime,PULocationID
0,2023-01-01 00:26:10,166
1,2023-01-01 00:51:03,24
2,2023-01-01 00:35:12,223
3,2023-01-01 00:13:14,41
4,2023-01-01 00:33:04,41


,lpep_pickup_datetime,PULocationID
0,2023-02-01 00:46:22,74
1,2023-02-01 00:05:09,216
2,2023-02-01 00:03:47,7
3,2023-01-31 23:30:56,74
4,2023-02-01 00:15:05,82
